In [1]:
from neo4j import GraphDatabase
import os
from dotenv import load_dotenv
load_dotenv()

True

Neo4j connection

In [14]:
URI = "neo4j://127.0.0.1:7687"   # exactly from Desktop
USER = "neo4j"
PASSWORD = "BSW@19#neo4j"  # the password you set in Desktop

driver = GraphDatabase.driver(
    URI,
    auth=(USER, PASSWORD)
)


Test Connection

In [15]:
with driver.session(database="neo4j") as session:
    result = session.run("RETURN 1 AS ok")
    print(result.single()["ok"])

AuthError: {neo4j_code: Neo.ClientError.Security.Unauthorized} {message: The client is unauthorized due to authentication failure.} {gql_status: 42NFF} {gql_status_description: error: syntax error or access rule violation - permission/access denied. Access denied, see the security logs for details.}

Create Nodes

In [9]:
def create_nodes(tx, node):
    query = f"""
    MERGE (n:{node['label']} {{id: $id}})
    SET n.name = $name,
        n.lang = $lang,
        n.description = $description
    """
    tx.run(
        query,
        id=node["node_id"],
        name=node.get("name"),
        lang=node.get("lang"),
        description=node.get("description")
    )

with driver.session() as session:
    for _, row in tqdm(nodes_df.iterrows(), total=len(nodes_df)):
        session.execute_write(create_nodes, row)

  0%|          | 0/55 [00:02<?, ?it/s]


AuthError: {neo4j_code: Neo.ClientError.Security.Unauthorized} {message: The client is unauthorized due to authentication failure.} {gql_status: 42NFF} {gql_status_description: error: syntax error or access rule violation - permission/access denied. Access denied, see the security logs for details.}

Create Relationships

In [ ]:
def create_relationship(tx, rel):
    query = f"""
    MATCH (a {{id: $start_id}})
    MATCH (b {{id: $end_id}})
    MERGE (a)-[r:{rel['type']}]->(b)
    """
    tx.run(
        query,
        start_id=rel["start_id"],
        end_id=rel["end_id"]
    )

with driver.session() as session:
    for _, row in tqdm(rels_df.iterrows(), total=len(rels_df)):
        session.execute_write(create_relationship, row)

Verify Graph

In [ ]:
with driver.session() as session:
    result = session.run("""
        MATCH (n)
        RETURN labels(n)[0] AS label, count(*) AS count
    """)
    for r in result:
        print(r)